# pathlib & File I/O

`pathlib.Path` gives you an object-oriented interface to the filesystem that composes naturally with Python's string and I/O ecosystem. Pair it with the `json`, `csv`, `pickle`, `tempfile`, and `shutil` modules for a complete toolkit for reading, writing, and managing files without touching a single `os.path` call.

**What's inside:** `Path` construction and navigation, reading/writing text and bytes, `glob`/`rglob`, JSON, CSV, pickle, temporary files, and `shutil`.

**Learn more:** [pathlib](https://docs.python.org/3/library/pathlib.html) · [json](https://docs.python.org/3/library/json.html) · [csv](https://docs.python.org/3/library/csv.html) · [pickle](https://docs.python.org/3/library/pickle.html)

## 1. Path construction and properties

In [1]:
from pathlib import Path

p = Path('/usr/local/bin/python3')
print(p.name)       # 'python3'
print(p.stem)       # 'python3'
print(p.suffix)     # '' (no extension)
print(p.parent)     # /usr/local/bin
print(p.parts)      # ('/', 'usr', 'local', 'bin', 'python3')

# / operator builds paths
base = Path('/tmp')
child = base / 'myapp' / 'data.json'
print(child)        # /tmp/myapp/data.json
print(child.suffix) # '.json'

python3
python3

\usr\local\bin
('\\', 'usr', 'local', 'bin', 'python3')
\tmp\myapp\data.json
.json


In [2]:
from pathlib import Path

p = Path('some/relative/path.txt')
print(p.is_absolute())          # False
print(p.resolve())              # absolute path from cwd
print(p.with_suffix('.csv'))    # some/relative/path.csv
print(p.with_name('other.txt')) # some/relative/other.txt

False
C:\Users\saxma\devel\cool_things_python_can_do\some\relative\path.txt
some\relative\path.csv
some\relative\other.txt


## 2. Reading and writing files

In [3]:
from pathlib import Path
import tempfile, os

tmp = Path(tempfile.mkdtemp())

# write and read text
file = tmp / 'hello.txt'
file.write_text('Hello, pathlib!\nLine two.\n', encoding='utf-8')
print(file.read_text(encoding='utf-8'))

# write and read bytes
binary = tmp / 'data.bin'
binary.write_bytes(b'\x00\x01\x02\x03')
print(binary.read_bytes())

# open() works just like the built-in
with file.open(encoding='utf-8') as f:
    for line in f:
        print(repr(line), end='')

Hello, pathlib!
Line two.

b'\x00\x01\x02\x03'
'Hello, pathlib!\n''Line two.\n'

## 3. Navigating the filesystem

In [4]:
from pathlib import Path

cwd = Path('.')

# iterdir: immediate children
for child in cwd.iterdir():
    kind = 'dir' if child.is_dir() else 'file'
    print(f'{kind:4}  {child.name}')

dir   .claude
dir   .git
file  .gitignore
file  .python-version
dir   .venv
file  01 - Python Standard Library.ipynb
file  02 - Python Lists.ipynb
file  03 - Python Dictionaries.ipynb
file  04 - Generators and Itertools.ipynb
file  05 - Context Managers.ipynb
file  06 - Object-Oriented Python.ipynb
file  07 - Type Hints.ipynb
file  08 - Concurrency.ipynb
file  09 - Functional Programming.ipynb
file  10 - Regular Expressions.ipynb
file  11 - Structural Pattern Matching.ipynb
file  12 - pathlib and File IO.ipynb
file  13 - Logging & Debugging.ipynb
file  14 - Testing with pytest.ipynb
file  15 - Building CLI Tools.ipynb
file  16 - Database Access.ipynb
file  17 - Web Requests and Scraping.ipynb
file  18 - Data Analysis with pandas.ipynb
file  19 - Numerical Computing with NumPy.ipynb
file  20 - Visualization with Matplotlib.ipynb
file  21 - Scientific Computing with SciPy.ipynb
file  22 - Machine Learning with scikit-learn.ipynb
file  CLAUDE.md
file  LICENSE
file  pyproject.toml
file  RE

In [5]:
from pathlib import Path

cwd = Path('.')

# glob: shell-style pattern matching
notebooks = list(cwd.glob('*.ipynb'))
print(f'{len(notebooks)} notebooks')
for nb in sorted(notebooks)[:5]:
    print(' ', nb.name)

# rglob: recursive glob (searches subdirectories too)
all_py = list(cwd.rglob('*.py'))
print(f'\n{len(all_py)} .py files')

22 notebooks
  01 - Python Standard Library.ipynb
  02 - Python Lists.ipynb
  03 - Python Dictionaries.ipynb
  04 - Generators and Itertools.ipynb
  05 - Context Managers.ipynb

5750 .py files


## 4. Creating and removing paths

In [6]:
from pathlib import Path
import tempfile

tmp = Path(tempfile.mkdtemp())

# mkdir with parents and exist_ok
nested = tmp / 'a' / 'b' / 'c'
nested.mkdir(parents=True, exist_ok=True)
print(nested.exists(), nested.is_dir())

# write a file then remove it
f = nested / 'test.txt'
f.write_text('hello')
print(f.exists())     # True
f.unlink()            # delete file
print(f.exists())     # False

# remove empty directory
nested.rmdir()
print(nested.exists())

True True
True
False
False


## 5. JSON

In [7]:
import json, tempfile
from pathlib import Path

tmp = Path(tempfile.mkdtemp())
path = tmp / 'data.json'

data = {
    'users': [
        {'name': 'Alice', 'age': 32, 'active': True},
        {'name': 'Bob',   'age': 25, 'active': False},
    ],
    'count': 2,
}

# write
path.write_text(json.dumps(data, indent=2), encoding='utf-8')

# read back
loaded = json.loads(path.read_text(encoding='utf-8'))
print(loaded['users'][0]['name'])  # Alice
print(type(loaded['users']))       # <class 'list'>

Alice
<class 'list'>


## 6. CSV

In [8]:
import csv, tempfile
from pathlib import Path

tmp = Path(tempfile.mkdtemp())
path = tmp / 'people.csv'

rows = [
    {'name': 'Alice', 'age': '32', 'city': 'London'},
    {'name': 'Bob',   'age': '25', 'city': 'Paris'},
    {'name': 'Carol', 'age': '28', 'city': 'Berlin'},
]

# write
with path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['name', 'age', 'city'])
    writer.writeheader()
    writer.writerows(rows)

# read back
with path.open(encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(row)

{'name': 'Alice', 'age': '32', 'city': 'London'}
{'name': 'Bob', 'age': '25', 'city': 'Paris'}
{'name': 'Carol', 'age': '28', 'city': 'Berlin'}


## 7. pickle: serialise any Python object

In [9]:
import pickle, tempfile
from pathlib import Path

tmp = Path(tempfile.mkdtemp())
path = tmp / 'model.pkl'

# any Python object can be pickled
obj = {
    'weights': [0.1, 0.5, -0.3],
    'labels':  {'cat': 0, 'dog': 1},
    'trained': True,
}

# write binary
path.write_bytes(pickle.dumps(obj))

# read back
loaded = pickle.loads(path.read_bytes())
print(loaded['labels'])    # {'cat': 0, 'dog': 1}
print(type(loaded))        # <class 'dict'>

{'cat': 0, 'dog': 1}
<class 'dict'>


## 8. tempfile: safe temporary files and directories

In [10]:
import tempfile
from pathlib import Path

# NamedTemporaryFile: auto-deleted on close
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, encoding='utf-8') as f:
    f.write('temporary content')
    tmp_path = Path(f.name)

print(tmp_path.read_text(encoding='utf-8'))
tmp_path.unlink()   # clean up manually when delete=False

# TemporaryDirectory: cleaned up on exit from the with block
with tempfile.TemporaryDirectory() as tmpdir:
    p = Path(tmpdir) / 'scratch.txt'
    p.write_text('hello')
    print(p.read_text())
# directory and its contents are deleted here

temporary content
hello


## 9. shutil: high-level file operations

In [11]:
import shutil, tempfile
from pathlib import Path

tmp = Path(tempfile.mkdtemp())
src = tmp / 'original.txt'
src.write_text('source content')

# copy file
dst = tmp / 'copy.txt'
shutil.copy2(src, dst)          # copy2 preserves metadata
print(dst.read_text())

# copy entire directory tree
(tmp / 'subdir' / 'nested').mkdir(parents=True)
(tmp / 'subdir' / 'nested' / 'a.txt').write_text('a')
(tmp / 'subdir' / 'b.txt').write_text('b')
shutil.copytree(tmp / 'subdir', tmp / 'subdir_copy')
print(list((tmp / 'subdir_copy').rglob('*')))

# remove entire directory tree
shutil.rmtree(tmp / 'subdir_copy')

source content
[WindowsPath('C:/Users/saxma/AppData/Local/Temp/tmpbwzn3s6t/subdir_copy/b.txt'), WindowsPath('C:/Users/saxma/AppData/Local/Temp/tmpbwzn3s6t/subdir_copy/nested'), WindowsPath('C:/Users/saxma/AppData/Local/Temp/tmpbwzn3s6t/subdir_copy/nested/a.txt')]
